# I22 server and chunked-HDF performance

> **Supplementary poster/testing notebook.** Original processing date: **2026-09-15**; MoDaCor version: **1.8.0**. This is not part of the core I22 correction example.

This notebook measures the real 2.48 MP I22 SAXS/WAXS data path from a new detector input to corrected, azimuthally integrated $I(q)$. It compares fresh-session full processing with sample-only partial reruns, then measures the persistent direct-slice HDF chunk workflow.

Preprocessing, server start-up, session construction, the chunk schema pilot, and plot generation are outside the reported timings. The runtime's processing-only clock and the client's end-to-end clock are both retained in the exported CSV files.

In [ ]:
from pathlib import Path
import atexit
import os
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'example_utils.py').is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError('Start Jupyter from MoDaCor_examples or one of its subdirectories.')

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir('DLS/I22')
sys.path.insert(0, str(PROJECT_DIR))
POSTER_DIR = PROJECT_DIR / 'supplementary' / 'poster_2026'
sys.path.insert(0, str(POSTER_DIR))
(PROJECT_DIR / 'work' / '.matplotlib').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_DIR / 'work' / '.matplotlib'))

import hdf5plugin
from IPython.display import display
from modacor.client import LocalRuntimeServer

from i22_helpers import prepare_inputs
from i22_performance import (
    BenchmarkSettings,
    benchmark_chunked_hdf,
    benchmark_server,
    combine_frames,
    new_run_dir,
    plot_overview,
    save_results,
)

## Configuration

The defaults are the poster-quality run: both detectors, three independent full sessions, six partial reruns, and all forty 10-frame chunks per detector. `FRAME_COUNT` limits only the chunk workload; whole-measurement server runs always use all 100 frames. For a quick functional check, set the repeat counts to one, `MEASUREMENT_LIMIT = 2`, and `FRAME_COUNT = 20`.

In [ ]:
RUN_SERVER = True
RUN_CHUNKED_HDF = True
DETECTORS = ('SAXS', 'WAXS')
MEASUREMENT_LIMIT = 4
FULL_REPEATS = 3
PARTIAL_REPEATS = 6
FRAME_COUNT = 100
CHUNK_SIZE = 10
WRITE_SERVER_HDF = True

ABSOLUTE_INTENSITY_FACTOR = 3.8e-15  # provisional DAWN-derived value
TRANSMISSION_REFERENCE_FILE = PROJECT_DIR / 'data' / 'i22-977723.nxs'
OVERWRITE_PREPROCESSED = False
SERVER_HOST = '127.0.0.1'
SERVER_PORT = 8902

settings = BenchmarkSettings(
    detectors=DETECTORS,
    measurement_limit=MEASUREMENT_LIMIT,
    full_repeats=FULL_REPEATS,
    partial_repeats=PARTIAL_REPEATS,
    frame_count=FRAME_COUNT,
    chunk_size=CHUNK_SIZE,
    write_server_hdf=WRITE_SERVER_HDF,
)
run_dir = new_run_dir(PROJECT_DIR, label='i22-server-hdf')
run_dir

## Prepare inputs and start the runtime

This work is intentionally untimed. Existing preprocessed files are reused, so the benchmark concerns pipeline execution rather than beamline-file compatibility reshaping.

In [ ]:
inputs = prepare_inputs(
    PROJECT_DIR,
    transmission_reference_file=TRANSMISSION_REFERENCE_FILE,
    absolute_intensity_factor=ABSOLUTE_INTENSITY_FACTOR,
    overwrite=OVERWRITE_PREPROCESSED,
)

server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=run_dir / 'modacor_server.log',
    environment={'HDF5_PLUGIN_PATH': hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")
print(f'Results: {run_dir}')

## Whole-measurement server workflow

Each `initial/full` observation uses a fresh session. A separate seeded session measures `sample rerun`; static detector geometry and corrected background remain cached, while every sample-dependent descendant is recomputed. Source registration is recorded separately and included in `workflow_s`.

In [ ]:
server_timings, server_steps = (
    benchmark_server(client, inputs, run_dir, settings)
    if RUN_SERVER else (None, None)
)

## Direct-slice chunked-HDF workflow

The schema-establishing pilot is not reported. Each measured request reads only its selected detector and normalization slices, executes the sample-dependent correction chain, and publishes the integrated result into one persistent HDF output. Finalization time is saved separately.

In [ ]:
chunk_timings, chunk_steps, finalize_timings = (
    benchmark_chunked_hdf(client, inputs, run_dir, settings)
    if RUN_CHUNKED_HDF else (None, None, None)
)

## Export and inspect

Intervals are empirical 10th--90th percentiles; bars show medians. Raw observations and per-step durations remain available for alternative poster layouts and later comparisons with MOUSE.

In [ ]:
timings = combine_frames(frame for frame in (server_timings, chunk_timings) if frame is not None)
step_timings = combine_frames(frame for frame in (server_steps, chunk_steps) if frame is not None)
if finalize_timings is None:
    import pandas as pd
    finalize_timings = pd.DataFrame()

summary = save_results(run_dir, settings, inputs, timings, step_timings, finalize_timings)
display(summary.round(3))
figure = plot_overview(summary, run_dir)

In [ ]:
top_steps = (
    step_timings.groupby(['workflow', 'detector', 'run_kind', 'step_id'], as_index=False)['seconds']
    .median()
    .sort_values('seconds', ascending=False)
    .groupby(['workflow', 'detector', 'run_kind'], as_index=False)
    .head(5)
)
display(top_steps.round(4))
print(f'CSV, metadata, and PNG/SVG/PDF figure written to {run_dir}')

In [ ]:
server.stop()